In [68]:
import requests as req
from bs4 import BeautifulSoup as bs
import shutil, textwrap


In [69]:
RESET = "\033[0m"
BOLD = "\033[1m"
DIM = "\033[2m"
CYAN = "\033[36m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
WHITE = "\033[37m"
RED = "\033[31m"

In [2]:
ANIME_BASE_URL = "https://otakudesu.blog"
ANIME_LIST = ANIME_BASE_URL + "/anime-list"

In [15]:
def extractAnimeList():
    getList = req.get(ANIME_LIST)
    soup = bs(getList.text, "html.parser")
    elClass = soup.find_all(class_="hodebgst")
    animes = []
    for element in elClass:
        anime = {
            "title": element.text.strip(),
            "path": element.get("href")
        }
        animes.append(anime)
    return animes

In [63]:
def parseAnimeDetail(html):
    soup = bs(html, "html.parser")

    anime = {}

    title = soup.select_one(".jdlrx h1")
    anime["title"] = title.text.strip() if title else None

    sub_title = soup.select_one(".subheading h2")
    anime["sub_title"] = (
        sub_title.text.strip()
        if sub_title else None
    )

    poster = soup.select_one(".fotoanime img")
    anime["poster"] = poster.get("src") if poster else None

    info = {}

    for p in soup.select(".infozingle p"):
        text = p.text.strip()

        if ":" in text:
            key, value = text.split(":", 1)
            info[key.strip()] = value.strip()

    anime["info"] = info

    synopsis = soup.select_one(".sinopc")
    anime["synopsis"] = (
        synopsis.text.strip()
        if synopsis else None
    )

    anime["genres"] = [
        a.text.strip()
        for a in soup.select(".infozingle a[rel='tag']")
    ]

    anime["episodes"] = []

    for li in soup.select(".episodelist ul li"):
        a = li.select_one("a")

        if a:
            anime["episodes"].append({
                "title": a.text.strip(),
                "path": a.get("href")
            })

    return anime


In [64]:
def getAnimeDetail(url):
    response = req.get(url)
    return parseAnimeDetail(response.text)

In [66]:
detail = getAnimeDetail("https://otakudesu.blog/anime/ogairu-season-3-sub-indo")

In [71]:
def truncate(text, max_length):
    if len(text) <= max_length:
        return text

    return text[:max_length - 3] + "..."

In [76]:
def displayAnimeDetail(anime):
    width = shutil.get_terminal_size().columns
    width = min(width, 100)

    print()

    # Header
    print(f"{BOLD}{CYAN}┌{'─' * (width - 2)}┐{RESET}")
    title = truncate(anime.get("title"), width - 6)

    print(
        f"{BOLD}{CYAN}│{RESET} "
        f"{BOLD}{WHITE}{title}{RESET}"
        f"{' ' * max(0, width - len(title) - 3)}"
        f"{BOLD}{CYAN}│{RESET}"
    )

    print(f"{BOLD}{CYAN}└{'─' * (width - 2)}┘{RESET}")

    # Basic info
    print(f"\n{BOLD}{BLUE}:: Information{RESET}")
    info = anime.get("info", {})
    fields = [
        ("Judul", info.get("Judul")),
        ("Japanese", info.get("Japanese")),
        ("Skor", info.get("Skor")),
        ("Tipe", info.get("Tipe")),
        ("Status", info.get("Status")),
        ("Total Episode", info.get("Total Episode")),
        ("Durasi", info.get("Durasi")),
        ("Tanggal Rilis", info.get("Tanggal Rilis")),
        ("Studio", info.get("Studio")),
        ("Produser", info.get("Produser")),
    ]

    for label, value in fields:
        if not value:
            continue

        value = truncate(value, width - 23)

        print(
            f"  {DIM}{label:<14}{RESET} "
            f"{WHITE}{value}{RESET}"
        )

    # Genre
    genres = anime.get("genres", [])

    if genres:
        print(
            f"  {DIM}{'Genre':<14}{RESET} "
            f"{GREEN}{', '.join(genres)}{RESET}"
        )

    # Synopsis
    print(f"\n{BOLD}{BLUE}:: Synopsis{RESET}")

    synopsis = anime.get("synopsis") or "-"
    for line in textwrap.wrap(synopsis, width=width - 4):
        print(f"  {WHITE}{line}{RESET}")

    # Episodes
    episodes = anime.get("episodes", [])

    print(
        f"\n{BOLD}{BLUE}:: Episodes "
        f"{DIM}({len(episodes)}){RESET}"
    )

    column_width = width // 2

    for i in range(0, len(episodes), 2):
        left = episodes[i]
        right = episodes[i + 1] if i + 1 < len(episodes) else None

        left_title = truncate(
            left.get("title"),
            column_width - 8
        )

        left_text = (
            f"{YELLOW}{i + 1:>3}{RESET} "
            f"{left_title}"
        )

        if right:
            right_title = truncate(
                right.get("title"),
                column_width - 8
            )

            right_text = (
                f"{YELLOW}{i + 2:>3}{RESET} "
                f"{right_title}"
            )
        else:
            right_text = ""

        plain_left = f"{i + 1:>3} {left_title}"
        padding = max(
            1,
            column_width - len(plain_left)
        )

        print(
            f"  {left_text}"
            f"{' ' * padding}"
            f"{right_text}"
        )

    print(
        f"{GREEN}✓{RESET} "
        f"{len(episodes)} episodes"
    )

    if anime.get("poster"):
        print(
            f"{DIM}Poster:{RESET} "
            f"{anime['poster']}"
        )

    print()


In [ ]:
displayAnimeDetail(detail)

In [70]:
def displayAnimeList(anime_list):
    width = shutil.get_terminal_size().columns
    column_width = width // 2

    print("\033[1;34mAnime List:\033[0m\n")

    for i in range(0, len(anime_list), 2):
        left = anime_list[i]
        right = anime_list[i + 1] if i + 1 < len(anime_list) else None

        left_title = truncate(
            left["title"],
            column_width - 8
        )

        left_text = f"{i + 1:>3}. {left_title}"

        if right:
            right_title = truncate(
                right["title"],
                column_width - 8
            )
            right_text = f"{i + 2:>3}. {right_title}"
        else:
            right_text = ""

        print(f"{left_text:<{column_width}}{right_text}")

    print(f"\n\033[2m{len(anime_list)} results\033[0m")


In [27]:
animeList = extractAnimeList()

In [28]:
displayAnimeList(animeList)


Anime List:

  1. #Compass 2.0: Sentou Setsuri ...     2. 100-man no Inochi no Ue ni Or...
  3. 100-man no Inochi no Ue ni Or...     4. 11eyes
  5. 12-sai.: Chicchana Mune no To...     6. 16bit Sensation: Another Layer
  7. 2.43: Seiin Koukou Danshi Vol...     8. 2.5-jigen no Ririsa
  9. 22/7                                10. 29-sai Dokushin Chuuken Bouke...
 11. 3-gatsu no Lion                     12. 3-gatsu no Lion Season 2
 13. 3D Kanojo: Real Girl                14. 3D Kanojo: Real Girl Season 2
 15. 4-nin wa Sorezore Uso wo Tsuku      16. 86
 17. 86 Season 2                         18. 91 Days
 19. A-Channel                           20. A-Rank Party wo Ridatsu shita...
 21. AICO Incarnation                    22. Absolute Duo
 23. ACCA: 13-ku Kansatsu-ka             24. Accel World
 25. Acchi Kocchi                        26. Acro Trip
 27. Adachi to Shimamura                 28. Aharen-san wa Hakarenai
 29. Aharen-san wa Hakarenai Season 2    30. Ahiru no Sora
 31. Aho Girl   

In [46]:
def searchAnime(keyword):
    results = []
    for anime in animeList:
        if keyword in anime["title"]:
            results.append(anime)
    displayAnimeList(results)

In [47]:
searchAnime("one")

Anime List:

  1. Campione                             2. Dr. Stone
  3. Dr. Stone Season 3                   4. Dr. Stone Season 3 Part 2
  5. Dr. Stone Season 4                   6. Dr. Stone Season 4 Part 2
  7. Dr. Stone: Science Future Par...     8. Dr. Stone Season 2
  9. Fairy Gone                          10. Fairy Gone Season 2
 11. Honey Lemon Soda                    12. Isekai wa Smartphone to Tomo ni
 13. Isekai wa Smartphone to Tomo ...    14. Kuroneko to Majo no Kyoushits...
 15. Shimoneta                           16. Shironeko Project: Zero Chron...

16 results


In [88]:
def getAnimeStream():
    r = req.get("https://otakudesu.blog/episode/orgr-s3-episode-11-sub-indo")
    soup = bs(r.text, "html.parser")
    embedStream = soup.find(class_="responsive-embed-stream")
    srcStream = embedStream.find("iframe").get("src")
    print(srcStream)

In [89]:
getAnimeStream()

https://desustream.me/dstream/otakuplay/new/index.php?id=VUhMODhWOGt2a2JKYTk5SW1hVk1zbW9yZnpkMTFmTCtxQTBNZnRRL3plcz0=
